# 02 Lane Assignment

**Description:** Assign detected vehicles to logical traffic lanes using centroid-based heuristics.

**Objective:** Load detection results, map vehicles into lanes, and save lane assignment results into `outputs/lane_assignment/`.

## Step 1. Setup

We load the previous detection CSV and create a dedicated lane assignment output directory.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs root: {OUTPUT_ROOT}')

OUTPUT_DIR = OUTPUT_ROOT / 'lane_assignment'
os.makedirs(OUTPUT_DIR, exist_ok=True)
INPUT_CSV = OUTPUT_ROOT / 'vehicle_detection' / 'detected_vehicles.csv'
OUTPUT_CSV = OUTPUT_DIR / 'lane_assignments.csv'


## Step 2. Load Detection Results

If the prior notebook has not been run yet, we create a small mock table so the rest of the notebook remains executable.

In [ ]:
if INPUT_CSV.exists():
    detections = pd.read_csv(INPUT_CSV)
else:
    detections = pd.DataFrame([
        {'vehicle_id': 1, 'centroid_x': 120, 'centroid_y': 260},
        {'vehicle_id': 2, 'centroid_x': 280, 'centroid_y': 250},
        {'vehicle_id': 3, 'centroid_x': 430, 'centroid_y': 255},
    ])

detections.head()


## Step 3. Lane Assignment Logic

Replace the simple x-coordinate thresholds with polygon-based lane mapping if you want to mirror the production pipeline more closely.

In [ ]:
def assign_lane(centroid_x: float) -> str:
    if centroid_x < 160:
        return 'lane_1'
    if centroid_x < 320:
        return 'lane_2'
    if centroid_x < 480:
        return 'lane_3'
    return 'lane_4'


def build_lane_assignments(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    output['lane_id'] = output['centroid_x'].apply(assign_lane)
    return output

lane_df = build_lane_assignments(detections)
lane_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved lane assignments to {OUTPUT_CSV}')
lane_df.head()
